# Page 14 — the distill dividend, measured

Pages 5 to 9 measured what answer-time compute buys on 100 MATH-500 level 4-5 problems: one
chain, then a longer chain, then sixteen chains and a vote, then a verifier choosing between
them. This notebook puts a **reasoning-trained** model on the same hundred problems and asks
what the *training* knob bought instead, in the two coordinates the deck has used throughout:
**problems solved** and **tokens spent**.

Four arms, two sizes, one item set:

- `Qwen2.5-1.5B-Instruct` and `Qwen2.5-7B-Instruct`, sixteen sampled chains each, read the way
  pages 7 and 8 read them: the **majority vote**, which is realizable, and **coverage**, which
  is what a perfect judge would get.
- `DeepSeek-R1-Distill-Qwen-1.5B` and `-7B`, run the way anyone actually runs them: **one
  chain, and nothing selects between chains**. Four chains per item are generated because a
  mean over four is a tighter estimate of one chain's accuracy than one chain is; no vote, no
  oracle, no best-of-anything is ever applied to them.

DeepSeek's published table is not a number on this page's figure. It is a **prediction** that
section 6 states in advance and section 13 checks (gate 4).

Writes `out/page_14/`: `traces.jsonl`, `derived.json`, `manifest.json`. Shared plumbing lives
in [`harness.py`](harness.py), which ships alongside this notebook.


## 0. How to run this

| Mode | What it does | Items | Chains | Hardware | Time |
|---|---|---|---|---|---|
| **`PLAN`** *(default)* | everything except generation: pins the items, states the gates, prices the run | 100 | none | **none** | seconds |
| `SMOKE` | the same four arms, small, to price `FULL` and to test the cap | 10 | 4 / 2 | 16 GB GPU | ~30 min |
| `FULL` | the run the page ships | 100 | 16 / 4 | 16 GB GPU | overnight |
| `REPLAY` | every figure and gate from the published generations | 100 | 16 / 4 | **none** | ~1 min |

**`PLAN` is the default because opening a notebook and running every cell must not start a GPU
job.** `REPLAY` becomes the default in the published download, once there is a run to replay.

**`SMOKE` changes the item count and the chain count and never the caps.** The cap is what the
truncation gate measures, so a smoke run at a smaller cap would clear a gate the real run then
fails.

```bash
python -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
jupyter lab
```

The 7B arms load in 8-bit, which needs `bitsandbytes`; 7.6B in bf16 does not fit beside its
activations on a 16 GB card.


In [ ]:
# ---------------------------------------------------------------------------
# The knobs. Everything else is derived from these.
# ---------------------------------------------------------------------------
RUN_MODE = "PLAN"         # "PLAN" (no GPU, no network), "SMOKE", "FULL", "REPLAY"
SEED = 123                # pages 6 to 9's seed: it draws the items and seeds every group
DATASET = "math500_l45"

# The draw pages 6 and 7 published. Section 2 refuses to run against anything else: an
# unpaired comparison against the deck's measured spine is the one failure here that would
# not announce itself.
PINNED_ITEM_HASH = "9ab2b579204221a0c701e6c54ec2c041a63500f9d633ba208aebef8606c43155"

# ---- the four arms ---------------------------------------------------------
# Two families at two sizes, each at ITS OWN documented operating point rather than at a
# shared one:
#   T=0.3, top-p 0.95   what page 7's temperature sweep chose for voting on this band.
#   T=0.6, top-p 0.95   DeepSeek's usage recommendation for the distills. They also
#                       document greedy decoding as producing repetition loops, so a
#                       greedy "headline" arm would measure the loop rather than the model.
# The asymmetry is deliberate and it is disclosed in the figure's ledger. Running both
# families at one temperature would put one of them somewhere nobody runs it.
#
# `chains` is a best-of-N axis for the instructs and an ESTIMATOR's sample size for the
# distills. Nothing selects among the distill's four.
#
# `group` is how many chains of one problem go into a single generate call, and it is set by
# what the KV cache costs at that arm's cap and precision, not by taste.
ARMS = [
    dict(key="instruct-1.5B", family="instruct", size_b=1.5,
         model="Qwen/Qwen2.5-1.5B-Instruct", quant="bf16",
         temperature=0.3, top_p=0.95, cap=2048, chains=16, smoke_chains=4, group=8),
    dict(key="instruct-7B", family="instruct", size_b=7.0,
         model="Qwen/Qwen2.5-7B-Instruct", quant="8bit",
         temperature=0.3, top_p=0.95, cap=2048, chains=16, smoke_chains=4, group=4),
    dict(key="distill-1.5B", family="distill", size_b=1.5,
         model="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B", quant="bf16",
         temperature=0.6, top_p=0.95, cap=8192, chains=4, smoke_chains=2, group=4),
    dict(key="distill-7B", family="distill", size_b=7.0,
         model="deepseek-ai/DeepSeek-R1-Distill-Qwen-7B", quant="8bit",
         temperature=0.6, top_p=0.95, cap=8192, chains=4, smoke_chains=2, group=2),
]

N_ITEMS = 10 if RUN_MODE == "SMOKE" else 100
N_AXIS = [1, 2, 4, 8, 16]     # the instructs' best-of-N axis, drawn from one pile
N_RESAMPLES = 64              # random N-subsets per (item, N), page 7's axis rule
ANSWER_ALLOWANCE = 32         # tokens allowed to the forced answer, page 6's rule
BOOTSTRAP_REPS = 10_000       # item-level resamples for the paired difference

# ---- the gates, before the run ---------------------------------------------
# A chain that hit its cap is one WE cut. Above this share the arm is a censorship
# measurement rather than an accuracy one, and the answer is a higher cap and a re-run
# (page 6's F13/F59 precedent), not a footnote.
TRUNCATION_MAX = 0.10

TRACES_URL = None             # where REPLAY reads the published generations

assert RUN_MODE in ("PLAN", "SMOKE", "FULL", "REPLAY"), RUN_MODE
if RUN_MODE == "FULL":
    print("!" * 78)
    print("WARNING  FULL generates roughly five million tokens across four checkpoints, two")
    print("         of them 8-bit 7Bs. On a 16 GB card that is an overnight run. Section 7")
    print("         prices it, and SMOKE reprices it from measured rates.")
    print("!" * 78 + "\n")

for a in ARMS:
    a["n_chains"] = a["smoke_chains"] if RUN_MODE == "SMOKE" else a["chains"]
    a["config_id"] = f"t{a['temperature']:.2f}-c{a['cap']}-{a['quant']}"

print(f"mode={RUN_MODE}  dataset={DATASET}  items={N_ITEMS}")
for a in ARMS:
    print(f"  {a['key']:<14} {a['model']:<44} {a['config_id']:<18} "
          f"{a['n_chains']:>2} chains, groups of {a['group']}")

In [ ]:
import json
import math
import random
import textwrap
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import requests
import torch

# Shared plumbing, shipped beside this notebook.
import harness

# The inherited grading harness: text-in, text-out, no model dependency.
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
    load_math500_test,
    render_prompt,
    extract_final_candidate,
    grade_answer,
)

OUT_DIR = Path("out/page_14")
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

TRACES_PATH = OUT_DIR / "traces.jsonl"
DERIVED_PATH = OUT_DIR / "derived.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"

mean, median, wilson = harness.mean, harness.median, harness.wilson


def size_label(b):
    return f"{b:g}B"


def read_traces(path):
    """Every record in the log, last write per key winning. The log is append-only and a
    resumed group can legitimately write a key twice; counting one chain as two would
    understate coverage, which is the ceiling this page's whole comparison rests on."""
    latest = {}
    for line in Path(path).open(encoding="utf-8"):
        if not line.strip():
            continue
        r = json.loads(line)
        latest[(r["run_id"], r["model_id"], r["config_id"], r["problem_id"],
                r["sample_index"])] = r
    return list(latest.values())


def prior(page):
    """A previous page's derived.json, from its run directory or from the copy committed
    beside the web app. Returns None rather than raising: a missing prior costs a
    cross-check, and every cell that reads one says what it lost."""
    for p in (Path(f"out/page_{page:02d}/derived.json"),
              Path(f"../web/data/page-{page:02d}-derived.json")):
        if p.exists():
            return json.loads(p.read_text(encoding="utf-8")), str(p)
    return None, None


print("torch", torch.__version__)
print("harness", harness.fingerprint()["version"], harness.fingerprint()["sha256"][:12])

## 1. Grading, and where a chain's answer is

Inherited unchanged from ch03, exactly as pages 5 to 9 use it, so the numbers here are
commensurable with theirs.

One rule is new, because one family of models is new. **A chain that thinks out loud is graded
on what it says after it stops thinking.** A distill writes its reasoning inside
`<think>...</think>` and its answer after the close; an instruct chain has no such block, and
the rule then returns the whole trace, which is what pages 5 to 9 graded. Without it, a
discarded guess inside the thinking can be scored as the model's answer.


In [ ]:
THINK_OPEN, THINK_CLOSE = "<think>", "</think>"


def answer_text(trace):
    """What the grader reads: the text after the last close of a thinking block, or the
    whole trace when there is none."""
    parts = trace.rsplit(THINK_CLOSE, 1)
    return parts[-1] if len(parts) > 1 else trace


def extract(trace, fallback="number_then_full"):
    return extract_final_candidate(answer_text(trace), fallback=fallback)


CHECKS = [
    # (raw completion, ground truth, expected)
    (r"The answer is \boxed{9}.",                             "9",     True),
    (r"27",                                                   "9",     False),
    (r"\boxed{\frac{1}{2}}",                                  "0.5",   True),   # LaTeX vs decimal
    (r"\boxed{1,024}",                                        "1024",  True),   # thousands separator
    # cut at its cap, then forced to commit:
    ("Common denominator is 12, so 3/4 = 9/12 and 2/3 = 8/12. Now I need to subtr"
     "\n\nFinal answer: " + r"\boxed{\frac{17}{12}}",         r"\frac{17}{12}", True),
    # forced, but committed to the wrong thing: forcing must not rescue it.
    ("...\n\nFinal answer: " + r"\boxed{\frac{5}{12}}",       r"\frac{17}{12}", False),
    # a distill's shape: the answer after the close of the block.
    ("<think>\nTry 27. No, that double counts.\n</think>\n\nThe answer is "
     + r"\boxed{9}.",                                         "9",     True),
    # the case the rule exists for: the ONLY boxed expression is a discarded guess
    # inside the thinking, and the answer after it is written in prose.
    ("<think>\nMaybe " + r"\boxed{27}" + "?\n</think>\n\nSo the value is 9.",
     "9", True),
    # a distill cut off before it stopped thinking, then forced to commit. Our close
    # of the block is part of the intervention, not part of its reasoning.
    ("<think>\nThe denominator is 12 and the numerators are 9 and"
     "\n</think>\n\nFinal answer: " + r"\boxed{\frac{17}{12}}", r"\frac{17}{12}", True),
]
rows = [(raw, gt, expected, bool(grade_answer(extract(raw), gt)))
        for raw, gt, expected in CHECKS]
failures = [r for r in rows if r[3] != r[2]]
if failures:
    for raw, gt, expected, got in rows:
        flag = "ok " if got == expected else "!! "
        shown = raw if len(raw) < 58 else raw[:30] + " … " + raw[-25:]
        print(f"{flag}{shown!r:<64} -> {extract(raw)!r:<20} vs {gt!r:<18} {got}")
    raise AssertionError(f"{len(failures)} grader check(s) behaved unexpectedly")
print(f"grader preflight: {len(rows)}/{len(rows)} as expected")

# What the thinking rule is worth, shown rather than asserted.
_probe = "<think>\nMaybe " + r"\boxed{27}" + "?\n</think>\n\nSo the value is 9."
print(f"\nthe same chain, graded two ways against ground truth '9':")
print(f"  whole trace            -> {extract_final_candidate(_probe)!r:<8} "
      f"{bool(grade_answer(extract_final_candidate(_probe), '9'))}")
print(f"  after the last {THINK_CLOSE} -> {extract(_probe)!r:<8} "
      f"{bool(grade_answer(extract(_probe), '9'))}")

## 2. The item set

Pages 6 and 7's draw, re-derived here from the same seed and **asserted against the hash they
published**. Every comparison on this page is paired at the problem level because of this cell.


In [ ]:
math500 = load_math500_test(local_path=DATA_DIR / "math500_test.json")

band = sorted([r for r in math500 if int(r["level"]) in (4, 5)], key=lambda r: r["unique_id"])
random.Random(SEED).shuffle(band)
full_draw = band[:100]
ITEM_SET_HASH = harness.sha256_of(r["unique_id"] for r in full_draw)

assert ITEM_SET_HASH == PINNED_ITEM_HASH, (
    f"the draw hashes to {ITEM_SET_HASH[:12]} and pages 6 and 7 published "
    f"{PINNED_ITEM_HASH[:12]}. These are not the same hundred problems, so nothing here is "
    f"comparable to the measured spine. Check SEED, the band filter and the ch03 copy of "
    f"MATH-500.")

# SMOKE takes a prefix of the same draw rather than a new one, so a smoke item is a full-run
# item and the two runs can be read against each other.
items = full_draw[:N_ITEMS]
ITEM_IDS = [r["unique_id"] for r in items]
QUESTION = {r["unique_id"]: r["problem"] for r in items}
GROUND_TRUTH = {r["unique_id"]: r["answer"] for r in items}

print(f"{DATASET}: {len(items)} of {len(band)} available in the band")
print(f"full draw  sha256 {ITEM_SET_HASH[:12]}  == pages 6 and 7's published draw")
print(f"this run   {len(items)} items" + ("  (a prefix of that draw)" if N_ITEMS < 100 else ""))

## 3. The four arms

The instructs are the deck's own baseline at two sizes; the distills are the same two sizes
with reasoning trained in. Weights are pinned to the revisions page 6 ran wherever page 6's
manifest is on disk, so "the same model" means the same bytes rather than the same name.


In [ ]:
P6, P6_SRC = prior(6)
REVISION_PINS = {m["id"]: m.get("revision")
                 for m in (P6 or {}).get("manifest", {}).get("models", [])}
if REVISION_PINS:
    print(f"revision pins read from {P6_SRC}")
else:
    print("page 6's manifest is not on disk: the instruct arms are pinned by name only")

harness.table(
    "The arms",
    ["arm", "model", "quant", "T", "top-p", "cap", "chains", "what the chains are for",
     "revision pin"],
    [[a["key"], a["model"].split("/")[-1], a["quant"], a["temperature"], a["top_p"],
      a["cap"], a["n_chains"],
      "best-of-N axis" if a["family"] == "instruct" else "one accuracy, estimated 4 ways",
      (REVISION_PINS.get(a["model"]) or "-")[:12]] for a in ARMS],
    note="The distills never have anything selected from their chains. Their four are an "
         "estimator's sample size; the instructs' sixteen are the axis the page plots.")

ARM = {a["key"]: a for a in ARMS}
INSTRUCT_ARMS = [a for a in ARMS if a["family"] == "instruct"]
DISTILL_ARMS = [a for a in ARMS if a["family"] == "distill"]
SIZES = sorted({a["size_b"] for a in ARMS})
PAIRS = [(next(a for a in INSTRUCT_ARMS if a["size_b"] == s),
          next(a for a in DISTILL_ARMS if a["size_b"] == s)) for s in SIZES]
print("\npanels: " + ", ".join(f"{size_label(s)} ({i['key']} vs {d['key']})"
                               for s, (i, d) in zip(SIZES, PAIRS)))

## 4. The vote, coverage, and the estimator

Ported from page 7 sections 4 and 5 unchanged, except that grading reads `answer_text`. Two
chains have agreed when the inherited grader calls their answers equivalent, a tie is not a
vote, and a chain with no usable answer has not voted.

**`coverage` is the oracle**: the share of problems where *some* chain in the pile is right. It
is what a perfect judge would score, so it bounds every selection method page 8 measured, and
it is the line the distill's single chain is asked to beat.


In [ ]:
NO_ANSWER = None


def vote_key(text):
    """What a chain contributes to the tally, or None if it contributed nothing."""
    cand = extract(text, fallback="number_only")
    return cand if cand and cand.strip() else NO_ANSWER


_EQ_CACHE = {}


def equivalent(a, b):
    if a == b:
        return True
    k = (a, b) if a <= b else (b, a)
    if k not in _EQ_CACHE:
        try:
            # grade_answer is not documented as symmetric, so ask it both ways.
            _EQ_CACHE[k] = bool(grade_answer(a, b)) or bool(grade_answer(b, a))
        except Exception:
            _EQ_CACHE[k] = False
    return _EQ_CACHE[k]


def cluster(keys):
    """Answer strings grouped into equivalence classes, first-seen form as the label."""
    labels, order = [], []
    for k in keys:
        if k is NO_ANSWER:
            labels.append(NO_ANSWER)
            continue
        hit = next((lab for lab in order if equivalent(lab, k)), None)
        if hit is None:
            order.append(k)
            hit = k
        labels.append(hit)
    return labels, order


def tally(records):
    """One problem, N chains -> the row every figure reads."""
    gt = records[0]["ground_truth"]
    labels, _ = cluster([r["vote_key"] for r in records])
    counts = Counter([l for l in labels if l is not NO_ANSWER])
    ranked = counts.most_common()
    top = ranked[0][1] if ranked else 0
    winners = [a for a, c in ranked if c == top]
    tie = len(winners) > 1
    plurality = winners[0] if (winners and not tie) else None
    return dict(
        n_chains=len(records),
        distinct_answers=len(counts),
        plurality_correct=bool(plurality is not None and grade_answer(plurality, gt)),
        any_correct=any(r["correct"] for r in records),          # the oracle
        mean_chain_correct=mean([r["correct"] for r in records]),
        tie=tie,
        no_answer_rate=mean([l is NO_ANSWER for l in labels]),
        truncated_rate=mean([r["truncated"] for r in records]),
        # What the chains COST: the tokens the model itself wrote. Our forced answer is an
        # intervention and is counted separately, as page 7 counts it.
        thinking_tokens=sum(r["own_tokens"] for r in records),
    )

In [ ]:
SAMPLES = {}          # arm key -> problem_id -> [records], sorted by sample_index


def subsets_for(pid, n, size, repeats=N_RESAMPLES):
    """The draws one point averages over. Exhaustive when the pile holds exactly n."""
    if n >= size:
        return [list(range(size))]
    rng = random.Random(harness.sha256_of([str(SEED), pid, str(n), str(size)])[:15])
    return [rng.sample(range(size), n) for _ in range(repeats)]


def per_item_rows(arm_key, n):
    """Per problem, the mean over random n-subsets of that problem's pile. Page 7's axis
    rule: the whole N axis costs one pile rather than one sweep per point."""
    out = {}
    for pid, rs in SAMPLES.get(arm_key, {}).items():
        if len(rs) < n:
            continue
        ts = [tally([rs[i] for i in idx]) for idx in subsets_for(pid, n, len(rs))]
        out[pid] = {k: mean([t[k] for t in ts]) for k in
                    ("plurality_correct", "any_correct", "mean_chain_correct", "tie",
                     "truncated_rate", "thinking_tokens")}
    return out


def curve_point(arm_key, n):
    """One point on the instruct arm's best-of-N axis, with what it cost."""
    rows = per_item_rows(arm_key, n)
    if not rows:
        return None
    m = lambda k: mean([r[k] for r in rows.values()])
    n_items = len(rows)
    vote, cov = m("plurality_correct"), m("any_correct")
    return dict(
        n=n, n_items=n_items,
        vote=vote, vote_ci=list(wilson(vote * n_items, n_items)),
        coverage=cov, coverage_ci=list(wilson(cov * n_items, n_items)),
        single_chain=m("mean_chain_correct"), tie_rate=m("tie"),
        truncated_rate=m("truncated_rate"),
        # N chains of whatever length they wanted, which is what N actually costs.
        thinking_tokens=m("thinking_tokens"),
        n_draws=n_items * N_RESAMPLES)


def distill_point(arm_key):
    """pass@1 for an arm nothing selects from: the mean over its chains, per problem, then
    over problems.

    The interval is Wilson over ITEMS, each item contributing its mean over the arm's
    chains. Averaging chains first shrinks the per-item variance below the Bernoulli one
    the formula assumes, so the interval it returns is the wide reading, and n is the
    number of problems rather than the number of chains -- which is the honest denominator,
    since four chains of one problem are not four problems.
    """
    by_item = SAMPLES.get(arm_key, {})
    if not by_item:
        return None
    per_item = {pid: mean([r["correct"] for r in rs]) for pid, rs in by_item.items()}
    chains = [r for rs in by_item.values() for r in rs]
    n_items = len(per_item)
    acc = mean(list(per_item.values()))
    return dict(
        n_items=n_items, n_chains=len(chains),
        pass_at_1=acc, ci=list(wilson(acc * n_items, n_items)),
        # ONE chain is the thing the page draws: the cost of running it once.
        mean_tokens_per_chain=mean([r["own_tokens"] for r in chains]),
        median_tokens_per_chain=median([r["own_tokens"] for r in chains]),
        truncated_rate=mean([r["truncated"] for r in chains]),
        no_answer_rate=mean([r["vote_key"] is NO_ANSWER for r in chains]),
        per_item=per_item)


def paired_bootstrap(a, b, reps=BOOTSTRAP_REPS, seed=SEED):
    """The paired difference mean(a) - mean(b), resampled over problems.

    McNemar is the deck's paired test everywhere else and it does not apply here: it needs
    two booleans per item, and the distill's per-item score is a mean over four chains. So
    the pairing is kept and the test is an item-level bootstrap, seeded so it re-derives.
    """
    ids = sorted(set(a) & set(b))
    d = [a[i] - b[i] for i in ids]
    if not d:
        return None
    rng, n = random.Random(seed), len(d)
    draws = sorted(mean([d[rng.randrange(n)] for _ in range(n)]) for _ in range(reps))
    below = sum(1 for x in draws if x <= 0) / reps
    return dict(delta=mean(d), ci=[draws[int(0.025 * reps)], draws[int(0.975 * reps) - 1]],
                p=min(1.0, 2 * min(below, 1 - below)), n_items=n, reps=reps)

## 5. What the deck has already measured here, and what DeepSeek published

Two priors, and they do different jobs. Pages 6 and 7's numbers are on **these hundred
problems** and price this run. DeepSeek's table is on **other benchmarks entirely** and is only
ever read for its *direction*, which gate 4 checks.


In [ ]:
# Page 6's greedy curve per model, at the cap the instruct arms use. This is the cost model:
# a chain spends far less than its cap, and pricing the cap would put a week on the estimate.
P6_SPEND, P6_GREEDY = {}, {}
if P6:
    for curve in P6["figure_2"]["curves"]:
        pt = next((p for p in curve["points"] if p["budget"] == 2048), None)
        if pt:
            P6_SPEND[curve["model_id"]] = pt["mean_tokens"]
            P6_GREEDY[curve["model_id"]] = pt["accuracy"]
    harness.table(
        "Page 6, greedy, one chain, the same hundred problems",
        ["model", "solved", "mean tokens", "cap"],
        [[m.split("/")[-1], f"{P6_GREEDY[m]:.0%}", f"{P6_SPEND[m]:,.0f}", 2048]
         for m in P6_SPEND],
        note=f"from {P6_SRC}")
else:
    print("page 6's derived.json is not on disk: the cost model below falls back to the cap,")
    print("which over-prices the run, and the greedy cross-check in section 10 is skipped.")

P7, P7_SRC = prior(7)
P7_HANDOFF = (P7 or {}).get("handoff")
if P7_HANDOFF:
    print(f"\npage 7, Qwen2.5-3B-Instruct, {P7_HANDOFF['n_chains']} chains at "
          f"T={P7_HANDOFF['temperature']}, the same hundred problems:")
    print(f"  one chain {P7_HANDOFF['single_chain_accuracy']:.0%}   "
          f"vote {P7_HANDOFF['plurality_accuracy']:.0%}   "
          f"coverage {P7_HANDOFF['coverage']:.0%}")
    print(f"  -> on {P7_HANDOFF['thrown_away']:.0%} of problems a chain had the right answer "
          f"and the vote threw it away. That gap is what this page's oracle line draws.")

# DeepSeek-AI 2025 (arXiv:2501.12948, Nature 2025), Tables 4-5. NOT our measurement, NOT on
# our items, and never plotted here: two different benchmarks, both of which say the 7B
# distill beats the 1.5B. Gate 4 asks only whether that DIRECTION reproduces on our
# instrument, which is the most a published number on another population can be asked.
PUBLISHED = {
    1.5: {"aime24_pass1": 28.9, "math500_pass1": 83.9},
    7.0: {"aime24_pass1": 55.5, "math500_pass1": 92.8},
}
print("\nDeepSeek's published distill numbers, read for direction only:")
for s, v in PUBLISHED.items():
    print(f"  {s}B   AIME 2024 {v['aime24_pass1']:>5.1f}   full MATH-500 "
          f"{v['math500_pass1']:>5.1f}")
print("  Neither is this page's population: MATH-500 entire is easier than its levels 4-5,")
print("  and AIME is a different exam. Only 1.5B < 7B transfers as a prediction.")

## 6. The gates, and both branches of the page's copy, declared before the run

Written before anything is generated, so the run cannot choose the sentence it flatters.
Gate 3 is the blocking one: a truncated arm is a measurement of our cap.

The page's payoff sentence has two forms and both are written here. Section 13 selects one and
`derived.json` records which, so the page ships the branch the run licensed rather than the one
the plan hoped for.


In [ ]:
GATES = {
    "dominance": ("Does the distill's ONE chain beat the instruct's oracle over sixteen, at "
                  "the same size? Decided per size on the paired bootstrap: the lower end of "
                  "the 95% interval on (distill pass@1 - instruct coverage@16) above zero."),
    "cost": ("What does each point cost? Compared as tokens the model wrote per problem: one "
             "distill chain against sixteen instruct chains, and against the N where the "
             "instruct's oracle first reaches the distill's accuracy, if it ever does."),
    "truncation": (f"Under {TRUNCATION_MAX:.0%} of chains may hit their cap, per arm. Above "
                   f"it the arm measured the cap rather than the model: raise it and re-run."),
    "direction": ("Does 1.5B < 7B, DeepSeek's published direction, reproduce on our "
                  "instrument for the distills?"),
}

# Page copy, both branches, pre-written. No em dashes: the page's guard test forbids them in
# rendered copy. No provenance prose: the run's vitals belong in the figure's ledger.
COPY = {
    "dominance": (
        "Sixteen chains from the instruct model, with a perfect judge picking the winner from "
        "them, do not reach one chain from the distill of the same size. The training knob "
        "bought what answer-time selection cannot."),
    "reversal": (
        "On our hundred hardest problems the ordering flips: sixteen instruct chains, judged "
        "perfectly, reach past the distill's single chain. Same models, a different "
        "instrument, a different ranking, and that is page 15 arriving early."),
    "mixed": (
        "The two sizes disagree. At one size the distill's single chain clears a perfect "
        "judge over sixteen instruct chains and at the other it does not, so the dividend is "
        "a fact about a model rather than about a method."),
}

print("the gates, declared before anything is generated:\n")
for k, v in GATES.items():
    print(f"  {k}{'  (BLOCKING)' if k == 'truncation' else ''}")
    print(textwrap.fill(v, 88, initial_indent="    ", subsequent_indent="    "))
print("\nthe payoff sentence, both branches, written before the run:")
for k, v in COPY.items():
    print(f"\n  [{k}]")
    print(textwrap.fill(v, 88, initial_indent="    ", subsequent_indent="    "))

## 7. What this run will cost

Priced from page 6's measured spend for the instructs, and from an **assumption** for the
distills, which no page in this deck has ever run: they are trained to think for a long time,
and nothing on disk says how long on these problems. `SMOKE` replaces the assumption with a
measurement and reprices `FULL`, which is what `SMOKE` is for.


In [ ]:
# The one number here that is a guess, named so it cannot hide. R1-distill chains on
# competition maths commonly run into the thousands of tokens; 4,000 is the working figure
# and section 10 replaces it with the measured mean.
DISTILL_TOKENS_ASSUMED = 4000

# Order-of-magnitude batched rates on the author's RTX 5080, from pages 6 and 7's runs.
# Estimates, and SMOKE measures the real ones per arm.
RATES = {("bf16", 1.5): 600.0, ("bf16", 7.0): 200.0,
         ("8bit", 1.5): 200.0, ("8bit", 7.0): 90.0}


def expected_tokens_per_chain(arm):
    if arm["family"] == "instruct":
        spend = P6_SPEND.get(arm["model"])
        return spend if spend else arm["cap"]
    return DISTILL_TOKENS_ASSUMED


def price(arms, n_items):
    rows, total, hours = [], 0, 0.0
    for a in arms:
        per = expected_tokens_per_chain(a)
        toks = per * a["n_chains"] * n_items
        h = toks / RATES[(a["quant"], a["size_b"])] / 3600
        rows.append([a["key"], a["n_chains"], f"{per:,.0f}", f"{toks / 1e6:.2f} M",
                     f"{h:.1f} h", "measured" if a["family"] == "instruct" and
                     P6_SPEND.get(a["model"]) else "assumed"])
        total += toks
        hours += h
    rows.append(["total", "", "", f"{total / 1e6:.2f} M", f"{hours:.1f} h", ""])
    return rows, total, hours


ROWS, PLANNED_TOKENS, PLANNED_HOURS = price(ARMS, N_ITEMS)
harness.table(f"Projected cost, {RUN_MODE}, {N_ITEMS} items",
              ["arm", "chains", "tokens/chain", "tokens", "wall clock", "tokens/chain from"],
              ROWS, note="Wall clock is an estimate from prior runs on one card; whatever is "
                         "already in traces.jsonl comes off it.")

if RUN_MODE == "SMOKE":
    _rows, _t, _h = price([dict(a, n_chains=a["chains"]) for a in ARMS], 100)
    print(f"\nthe FULL run at this pricing: {_t / 1e6:.2f} M tokens, about {_h:.0f} h. "
          f"Section 10 reprices it from what this smoke run measured.")
if RUN_MODE == "PLAN":
    print("\nPLAN stops before generating. SMOKE next: it prices FULL from measured rates "
          "and\nmeasures the truncation the caps produce, which is the gate FULL has to "
          "clear.")

## 8. One group of chains

Page 7's generator, extended by the two things a reasoning model needs.

**The thinking block is opened for the distill, not left to chance.** DeepSeek document these
checkpoints as sometimes skipping the block, and a chain that never thinks is not the model
being measured. Where the chat template does not already open it, the prompt ends with
`<think>`, which we wrote and therefore never charge as a generated token.

**A chain the cap cut off is closed before it is forced to commit.** Pages 6 and 7 append
`Final answer: \boxed{` to a truncated chain; a distill cut off mid-thought is still inside an
open block, so the close comes first. It is the same intervention, spelled for a model that
marks where its thinking ends, and `forced` records every chain that got it.

**How "it stopped on its own" is decided under batching.** A batch returns every row padded to
the longest, so length carries no information: a chain finished on its own if and only if it
emitted an end-of-turn token.


In [ ]:
FORCE_STEM = "\n\nFinal answer: \\boxed{"      # the brace we open and it must close


def build_chat(arm, tok, question):
    """The exact string the model is given. Every arm gets ch03's stem and no system prompt,
    because a prompt difference between the families would be indistinguishable from the
    training difference this page exists to measure."""
    chat = tok.apply_chat_template([{"role": "user", "content": render_prompt(question)}],
                                   tokenize=False, add_generation_prompt=True)
    prefilled = False
    if arm["family"] == "distill" and not chat.rstrip().endswith(THINK_OPEN):
        chat += THINK_OPEN + "\n"
        prefilled = True
    return chat, prefilled


def force_stem_for(arm, own_text, opened):
    """What is appended to a chain the cap cut off. `opened` says the prompt already opened
    a thinking block, so a chain that never wrote the tag itself is still inside one."""
    if (arm["family"] == "distill" and (opened or THINK_OPEN in own_text)
            and THINK_CLOSE not in own_text):
        return "\n" + THINK_CLOSE + FORCE_STEM
    return FORCE_STEM


def cut_at_boxed_close(pieces):
    """We prefilled one open brace. Walk the continuation until it closes, tracking depth so
    nested LaTeX like \\frac{1}{2} is not cut in half."""
    depth, kept = 1, []
    for p in pieces:
        out = ""
        for ch in p:
            out += ch
            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    kept.append(out)
                    return kept, True
        kept.append(out)
    return kept, False


def group_seed(problem_id, cid, group_index):
    """A stable 63-bit seed per (problem, configuration, group), recorded on every record in
    the group so the group regenerates exactly."""
    return int(harness.sha256_of([str(SEED), problem_id, cid, str(group_index)])[:15], 16)


def think_span(pieces, opened):
    """(tokens the chain spent inside its thinking block, whether the block closed).
    (None, None) when there is no block at all, which is every instruct chain. The tag we
    prefilled is not counted: `opened` says it is there, and it is not the model's token."""
    run = ""
    for i, p in enumerate(pieces):
        run += p
        if THINK_CLOSE in run:
            return i + 1, True
    return (len(pieces), False) if (opened or THINK_OPEN in run) else (None, None)


@torch.inference_mode()
def sample_group(arm, question, k, seed, mdl, tok, eos_ids):
    """k chains of one problem in one generate call, forced to commit if the cap bound."""
    assert k >= 1, f"asked for {k} chains"
    chat, prefilled = build_chat(arm, tok, question)
    prompt_ids = tok(chat, add_special_tokens=False).input_ids
    inp = torch.tensor([prompt_ids], device=mdl.device)
    torch.manual_seed(seed)
    out = mdl.generate(
        input_ids=inp, attention_mask=torch.ones_like(inp), max_new_tokens=arm["cap"],
        # A checkpoint can ship sampling defaults in generation_config, so do_sample alone is
        # not enough: state every decoding parameter or the manifest records a setup that was
        # never configured.
        do_sample=True, temperature=arm["temperature"], top_p=arm["top_p"], top_k=None,
        num_return_sequences=k,
        pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id)

    chains = []
    for row in out:
        gen = row[inp.shape[1]:].tolist()
        stop = next((i for i, t in enumerate(gen) if t in eos_ids), None)
        hit_cap = stop is None
        gen = gen[:stop] if stop is not None else gen[:arm["cap"]]

        own_text = tok.decode(gen, skip_special_tokens=True)
        own_pieces = harness.incremental_token_strings(gen, tok)
        assert "".join(own_pieces) == own_text
        n_think, closed = think_span(own_pieces, prefilled)

        if not hit_cap:
            chains.append(dict(trace=own_text, own_tokens=len(gen), answer_tokens=0,
                               tokens_generated=len(gen), truncated=False, forced=False,
                               forced_suffix="", answer_closed=None,
                               think_tokens=n_think, think_closed=closed,
                               think_prefilled=prefilled))
            continue

        stem = force_stem_for(arm, own_text, prefilled)
        suffix_ids = tok(stem, add_special_tokens=False).input_ids
        suffix_pieces = harness.incremental_token_strings(suffix_ids, tok)
        assert "".join(suffix_pieces) == stem, (
            "the forced suffix does not survive a tokenise/detokenise round trip on this "
            "checkpoint; the intervention would land in the wrong place")
        # The forced answer is greedy at every temperature: it is our intervention rather
        # than the model's chain, and sampling it would put diversity where we injected it.
        forced_inp = torch.tensor([prompt_ids + gen + suffix_ids], device=mdl.device)
        ans = mdl.generate(input_ids=forced_inp, attention_mask=torch.ones_like(forced_inp),
                           max_new_tokens=ANSWER_ALLOWANCE, do_sample=False, temperature=None,
                           top_p=None, top_k=None,
                           pad_token_id=tok.pad_token_id if tok.pad_token_id is not None
                           else tok.eos_token_id)
        ans_ids = ans[0][forced_inp.shape[1]:].tolist()
        while ans_ids and ans_ids[-1] in eos_ids:
            ans_ids.pop()
        kept, closed_brace = cut_at_boxed_close(harness.incremental_token_strings(ans_ids, tok))

        chains.append(dict(trace="".join(own_pieces + suffix_pieces + kept),
                           own_tokens=len(gen), answer_tokens=len(kept),
                           # Our suffix is not the model's cost; the answer it wrote is.
                           tokens_generated=len(gen) + len(kept),
                           truncated=True, forced=True, forced_suffix=stem,
                           answer_closed=closed_brace,
                           think_tokens=n_think, think_closed=closed,
                           think_prefilled=prefilled))
    return chains

## 9. The sweep

One arm at a time, each model loaded and released before the next, because two 7B checkpoints
do not share a 16 GB card. Resumable: interrupt it and re-run, and only the missing groups are
made.

`REPLAY` reads the published generations instead, and nothing below knows the difference.


In [ ]:
RUN_ID = (f"p14distill-{RUN_MODE.lower()}-{DATASET}-n{len(items)}-{ITEM_SET_HASH[:8]}"
          f"-i{ARM['instruct-1.5B']['n_chains']}d{ARM['distill-1.5B']['n_chains']}")
ARM_TIMING, MODEL_META, GENERATED_HERE = {}, {}, 0

if RUN_MODE == "REPLAY":
    if TRACES_URL and not TRACES_PATH.exists():
        print(f"fetching published generations from {TRACES_URL}")
        TRACES_PATH.write_text(requests.get(TRACES_URL, timeout=600).text, encoding="utf-8")
    assert TRACES_PATH.exists(), (
        f"REPLAY needs published generations at {TRACES_PATH} and found none. Set TRACES_URL "
        f"in the knobs cell, or use RUN_MODE = 'SMOKE' to generate your own.")
    raw = read_traces(TRACES_PATH)
    assert raw, f"{TRACES_PATH} is empty"
    RUN_ID = Counter(r["run_id"] for r in raw).most_common(1)[0][0]
    raw = [r for r in raw if r["run_id"] == RUN_ID]
    want = set(ITEM_IDS)
    missing = want - {r["problem_id"] for r in raw}
    assert not missing, (
        f"the published traces miss {len(missing)} of this notebook's {len(want)} problems; "
        f"that file was made with a different seed, dataset or item count.")
    unknown = {r["arm"] for r in raw} - set(ARM)
    assert not unknown, (
        f"the published traces carry arms this notebook does not define: {sorted(unknown)}. "
        f"That file is a different experiment.")
    for r in raw:
        if r["problem_id"] in want:
            SAMPLES.setdefault(r["arm"], {}).setdefault(r["problem_id"], []).append(r)
    for key in SAMPLES:
        for pid in SAMPLES[key]:
            SAMPLES[key][pid].sort(key=lambda r: r["sample_index"])
        depth = min(len(v) for v in SAMPLES[key].values())
        ARM[key]["n_chains"] = min(ARM[key]["n_chains"], depth)
        MODEL_META[key] = {"id": ARM[key]["model"], "replayed": True}
    print(f"replaying {RUN_ID}")
    for a in ARMS:
        got = SAMPLES.get(a["key"], {})
        print(f"  {a['key']:<14} {len(got):>3} items x "
              f"{sorted({len(v) for v in got.values()})} chains")

elif RUN_MODE == "PLAN":
    print("PLAN: nothing generated. Sections 10 to 14 report what they can and skip the rest.")

else:
    device = get_device()
    DTYPE = harness.pick_dtype(device)
    STORE = harness.TraceStore(
        TRACES_PATH, RUN_ID, ("model_id", "config_id", "problem_id", "sample_index"))
    STORE.report()

    def ensure_arm(arm, mdl, tok, eos_ids):
        cid, k = arm["config_id"], arm["n_chains"]
        t0, n_gen, n_tok = time.time(), 0, 0
        bar = harness.progress(items, desc=f"{arm['key']:<14}", unit="item")
        for row in bar:
            pid = row["unique_id"]
            got = [STORE.get(arm["model"], cid, pid, s) for s in range(k)]
            for start in range(0, k, arm["group"]):
                size = min(arm["group"], k - start)
                if all(got[s] is not None for s in range(start, start + size)):
                    continue                      # this group is already on disk
                g_index = start // arm["group"]
                seed = group_seed(pid, cid, g_index)
                made = sample_group(arm, row["problem"], size, seed, mdl, tok, eos_ids)
                for j, g in enumerate(made):
                    s = start + j
                    got[s] = STORE.append({
                        "model_id": arm["model"], "config_id": cid, "arm": arm["key"],
                        "family": arm["family"], "size_b": arm["size_b"],
                        "quant": arm["quant"],
                        "problem_id": pid, "sample_index": s, "group_index": g_index,
                        "batch_seed": seed, "batch_size": size,
                        "level": int(row["level"]), "subject": row["subject"],
                        "question": row["problem"], "ground_truth": row["answer"],
                        "temperature": arm["temperature"], "top_p": arm["top_p"],
                        "chain_budget": arm["cap"],
                        # No per-token strings: this page animates nothing, and 3,600 chains
                        # of them would be a download in the hundreds of megabytes. The
                        # reconstruction invariant is asserted at generation time instead.
                        "trace": g["trace"],
                        "truncated": g["truncated"], "forced": g["forced"],
                        "forced_suffix": g["forced_suffix"],
                        "answer_closed": g["answer_closed"],
                        "think_tokens": g["think_tokens"],
                        "think_closed": g["think_closed"],
                        "think_prefilled": g["think_prefilled"],
                        "extracted_answer": extract(g["trace"]),
                        "vote_key": vote_key(g["trace"]),
                        "correct": bool(grade_answer(extract(g["trace"]), row["answer"])),
                        "own_tokens": g["own_tokens"], "answer_tokens": g["answer_tokens"],
                        "tokens_generated": g["tokens_generated"],
                    })
                    n_gen += 1
                    n_tok += g["tokens_generated"]
            assert all(g is not None for g in got), f"{arm['key']} {pid}: a chain is missing"
            SAMPLES.setdefault(arm["key"], {})[pid] = got
            bar.set_postfix(gen=n_gen, ktok=n_tok // 1000)
        secs = time.time() - t0
        ARM_TIMING[arm["key"]] = dict(seconds=secs, generated=n_gen, tokens=n_tok,
                                      tok_per_s=(n_tok / secs) if n_gen else None)
        return n_gen

    for arm in ARMS:
        print(f"\n{'=' * 78}\n{arm['key']}  {arm['model']}  {arm['quant']}  "
              f"T={arm['temperature']}  cap={arm['cap']}  {arm['n_chains']} chains\n")
        pin = REVISION_PINS.get(arm["model"])
        rev = harness.repo_revision(arm["model"])
        if pin and rev and pin != rev:
            print(f"!! revision {rev[:12]} is not page 6's pin {pin[:12]}: this arm is a "
                  f"different set of weights from the one the deck's other pages measured.")
        loaded = harness.load_model(arm["model"], device=device, dtype=DTYPE,
                                    load_in_8bit=arm["quant"] == "8bit", warmup_tokens=16)
        model, tokenizer = loaded["model"], loaded["tokenizer"]
        harness.selftest(tokenizer, verbose=False)
        eos = harness.end_of_turn_ids(tokenizer)
        MODEL_META[arm["key"]] = dict(
            id=arm["model"], label=arm["model"].split("/")[-1], quant=arm["quant"],
            params=loaded["params"], dtype=str(loaded["dtype"]), revision=rev,
            revision_pin=pin, tokenizer_sha256=harness.tokenizer_fingerprint(tokenizer),
            eos_ids=sorted(eos), warmup_tok_per_s=loaded["tok_per_s"])
        GENERATED_HERE += ensure_arm(arm, model, tokenizer, eos)
        del model, tokenizer, loaded
        harness.free_memory()
    print(f"\nsweep complete: {GENERATED_HERE:,} chains generated this session")

RECORDS = [r for rs in SAMPLES.values() for v in rs.values() for r in v]
HAVE_RUN = bool(RECORDS)
print(f"\n{len(RECORDS):,} chains in memory across {len(SAMPLES)} arm(s)")

## 10. Integrity, and the cap gate

Six checks. Two matter more than the rest here: **the cap gate**, which decides whether an arm
measured the model or measured our budget, and **the repricing**, which replaces section 7's
assumption about distill chain length with what the distills actually wrote.


In [ ]:
INTEGRITY, CAP_GATE = {}, {}
if not HAVE_RUN:
    print("nothing generated: the integrity checks and the cap gate have no run to check.")
else:
    keys = {(r["model_id"], r["config_id"], r["problem_id"], r["sample_index"])
            for r in RECORDS}
    print(f"1. one record per key ...................... {len(RECORDS):,} records, "
          f"{len(keys):,} keys")
    assert len(RECORDS) == len(keys), (
        "duplicate records in the trace bank. TraceStore has no write lock and two processes "
        "on one run_id silently double-count; deduplicate before trusting any number below.")

    depth = {k: sorted({len(v) for v in rs.values()}) for k, rs in SAMPLES.items()}
    print(f"2. every problem has its full pile ......... " +
          ", ".join(f"{k} {d}" for k, d in depth.items()))
    for a in ARMS:
        got = SAMPLES.get(a["key"], {})
        assert len(got) == len(items), f"{a['key']} covers {len(got)} of {len(items)} problems"
        assert min(depth[a["key"]]) >= a["n_chains"], f"{a['key']} is short of chains"

    print("\n3. what each arm did, and whether the cap bound it:")
    rows = []
    for a in ARMS:
        rs = [r for r in RECORDS if r["arm"] == a["key"]]
        trunc = mean([r["truncated"] for r in rs])
        CAP_GATE[a["key"]] = dict(
            cap=a["cap"], truncated_rate=trunc, threshold=TRUNCATION_MAX,
            passes=trunc <= TRUNCATION_MAX,
            forced_rate=mean([r["forced"] for r in rs]),
            unclosed_forced=mean([r["answer_closed"] is False for r in rs if r["forced"]])
                            if any(r["forced"] for r in rs) else 0.0,
            boxed_rate=mean(["\\boxed" in answer_text(r["trace"]) for r in rs]),
            no_answer_rate=mean([r["vote_key"] is None for r in rs]),
            mean_own_tokens=mean([r["own_tokens"] for r in rs]),
            median_own_tokens=median([r["own_tokens"] for r in rs]),
            think_closed_rate=mean([bool(r["think_closed"]) for r in rs]),
            mean_think_tokens=mean([r["think_tokens"] or 0 for r in rs]),
            n_chains=len(rs))
        g = CAP_GATE[a["key"]]
        rows.append([a["key"], a["cap"], f"{g['mean_own_tokens']:,.0f}",
                     f"{g['median_own_tokens']:,.0f}", f"{trunc:.0%}",
                     f"{g['boxed_rate']:.0%}", f"{g['think_closed_rate']:.0%}",
                     "ok" if g["passes"] else "CENSORED"])
    harness.table(
        "Per arm, over every chain",
        ["arm", "cap", "mean tokens", "median", "hit the cap", "boxed", "closed <think>",
         "cap gate"],
        rows,
        note=f"An arm over {TRUNCATION_MAX:.0%} is measuring the cap. Raise it and re-run; a "
             f"censored arm is not an accuracy measurement.")

    print("\n4. sampling sampled:")
    for a in ARMS:
        rs = SAMPLES[a["key"]]
        ident = mean([len({r["trace"] for r in v}) == 1 for v in rs.values() if len(v) > 1])
        print(f"   {a['key']:<14} identical chains on {ident:>4.0%} of problems")
        assert ident < 1.0 or a["n_chains"] == 1, (
            f"{a['key']} sampled at T={a['temperature']} and produced identical chains "
            f"everywhere; do_sample did not take effect")

    seeds = {(r["problem_id"], r["config_id"], r["group_index"]): r["batch_seed"]
             for r in RECORDS if r.get("batch_seed") is not None}
    print(f"5. one seed per group, groups distinct ..... {len(set(seeds.values()))} seeds for "
          f"{len(seeds)} groups")
    assert not seeds or len(set(seeds.values())) == len(seeds), "two groups share a seed"

    print("\n6. repricing the FULL run from what was measured:")
    for a in ARMS:
        per = CAP_GATE[a["key"]]["mean_own_tokens"]
        toks = per * a["chains"] * 100
        rate = (ARM_TIMING.get(a["key"], {}) or {}).get("tok_per_s") \
            or RATES[(a["quant"], a["size_b"])]
        src = "measured" if (ARM_TIMING.get(a["key"], {}) or {}).get("tok_per_s") else "prior"
        print(f"   {a['key']:<14} {per:>6,.0f} tok/chain  x {a['chains']:>2} chains x 100 "
              f"= {toks / 1e6:>5.2f} M   ~{toks / rate / 3600:>4.1f} h at {rate:,.0f} tok/s "
              f"({src})")
    if any(a["family"] == "distill" for a in ARMS):
        _m = mean([CAP_GATE[a["key"]]["mean_own_tokens"] for a in DISTILL_ARMS])
        print(f"\n   section 7 assumed {DISTILL_TOKENS_ASSUMED:,} tokens per distill chain and "
              f"they wrote {_m:,.0f}.")

    INTEGRITY = dict(n_records=len(RECORDS), n_generated_this_session=GENERATED_HERE,
                     cap_gate=CAP_GATE, timing=ARM_TIMING,
                     tokens_note="own_tokens is the model's own generation; the forced "
                                 "answer is counted separately and never on the cost axis")

## 11. The two panels

One panel per size. The instruct arm contributes two curves over N, each plotted at what N
chains actually cost; the distill contributes one point, at what one chain costs.


In [ ]:
CURVES, POINTS, PANELS = {}, {}, []
if not HAVE_RUN:
    print("nothing generated: no curves to build.")
else:
    for a in INSTRUCT_ARMS:
        axis = [n for n in N_AXIS if n <= a["n_chains"]]
        CURVES[a["key"]] = [p for n in axis if (p := curve_point(a["key"], n))]
        print(f"\n{a['key']}, {a['n_chains']} chains at T={a['temperature']}, "
              f"n={len(items)} problems")
        print(f"  {'N':>3}{'tokens':>10}{'vote':>9}{'coverage':>11}{'one chain':>12}"
              f"{'ties':>7}")
        for p in CURVES[a["key"]]:
            print(f"  {p['n']:>3}{p['thinking_tokens']:>10,.0f}{p['vote']:>9.0%}"
                  f"{p['coverage']:>11.0%}{p['single_chain']:>12.0%}{p['tie_rate']:>7.0%}")

    for a in DISTILL_ARMS:
        POINTS[a["key"]] = distill_point(a["key"])
        p = POINTS[a["key"]]
        print(f"\n{a['key']}, one chain per problem, estimated over {a['n_chains']} chains "
              f"on each of {p['n_items']} problems, T={a['temperature']}")
        print(f"  pass@1 {p['pass_at_1']:.0%}  [{p['ci'][0]:.0%}, {p['ci'][1]:.0%}]   "
              f"{p['mean_tokens_per_chain']:,.0f} tokens per chain "
              f"(median {p['median_tokens_per_chain']:,.0f})")
        print(f"  cut off {p['truncated_rate']:.0%}   no usable answer "
              f"{p['no_answer_rate']:.0%}")

    for inst, dist in PAIRS:
        curve, pt = CURVES.get(inst["key"], []), POINTS.get(dist["key"])
        top = curve[-1] if curve else None
        # Where sixteen instruct chains would have to reach to match one distill chain, and
        # what they cost getting there.
        reach = next((c for c in curve if pt and c["coverage"] >= pt["pass_at_1"]), None)
        PANELS.append(dict(
            size_b=inst["size_b"], instruct=inst["key"], distill=dist["key"],
            instruct_model=inst["model"], distill_model=dist["model"],
            curve=curve, point=pt,
            oracle_at_max=top["coverage"] if top else None,
            vote_at_max=top["vote"] if top else None,
            tokens_at_max=top["thinking_tokens"] if top else None,
            oracle_reaches_distill_at_n=reach["n"] if reach else None,
            cost_ratio=(top["thinking_tokens"] / pt["mean_tokens_per_chain"])
                       if (top and pt and pt["mean_tokens_per_chain"]) else None))

    harness.table(
        "The panels",
        ["size", "distill pass@1", "one chain", "instruct vote@N", "instruct oracle@N",
         "N chains cost", "cost ratio"],
        [[size_label(p["size_b"]), f"{p['point']['pass_at_1']:.0%}",
          f"{p['point']['mean_tokens_per_chain']:,.0f} tok",
          f"{p['vote_at_max']:.0%}", f"{p['oracle_at_max']:.0%}",
          f"{p['tokens_at_max']:,.0f} tok",
          f"{p['cost_ratio']:.1f}x" if p["cost_ratio"] else "-"] for p in PANELS],
        note="Cost is tokens the model wrote per problem. The oracle is what a perfect judge "
             "would score over the pile, so it bounds every selection method page 8 measured.")

## 12. The figure

The reference render. The page redraws it in the browser from `derived.json`, and the two must
show the same numbers.


In [ ]:
import matplotlib.pyplot as plt

if not HAVE_RUN:
    print("nothing generated: no figure.")
else:
    INK = dict(vote="#3b4a6b", oracle="#a2704a", distill="#b23", rule="#b23")
    fig, axes = plt.subplots(1, len(PANELS), figsize=(5.6 * len(PANELS), 4.4), squeeze=False)
    for ax, pan in zip(axes[0], PANELS):
        curve, pt = pan["curve"], pan["point"]
        xs = [c["thinking_tokens"] for c in curve]
        ax.plot(xs, [c["vote"] * 100 for c in curve], marker="o", ms=4.5, lw=2,
                color=INK["vote"], label="instruct, majority vote")
        ax.plot(xs, [c["coverage"] * 100 for c in curve], marker="s", ms=4, lw=1.6, ls="--",
                color=INK["oracle"], label="instruct, a perfect judge")
        for c in curve:
            ax.annotate(f"{c['n']}", (c["thinking_tokens"], c["coverage"] * 100),
                        textcoords="offset points", xytext=(0, 7), ha="center", fontsize=7,
                        color=INK["oracle"])
        ax.axhline(pt["pass_at_1"] * 100, lw=1, ls=":", color=INK["rule"], alpha=0.8)
        ax.errorbar([pt["mean_tokens_per_chain"]], [pt["pass_at_1"] * 100],
                    yerr=[[(pt["pass_at_1"] - pt["ci"][0]) * 100],
                          [(pt["ci"][1] - pt["pass_at_1"]) * 100]],
                    fmt="o", ms=8, lw=1.4, capsize=3, color=INK["distill"],
                    label="distill, one chain")
        ax.set_xscale("log")
        ax.set_xlabel("thinking tokens per problem")
        ax.set_ylabel("share of problems solved (%)")
        ax.set_ylim(0, 100)
        ax.grid(axis="y", alpha=0.25)
        ax.spines[["top", "right"]].set_visible(False)
        ax.legend(frameon=False, fontsize=8, loc="lower right")
        ax.set_title(size_label(pan["size_b"]), fontsize=10, loc="left", color="#333")
        # The ledger: the run's vitals inside the figure, which is where they belong.
        ledger = (f"{len(items)} problems, MATH-500 levels 4-5\n"
                  f"instruct  T={ARM[pan['instruct']]['temperature']}, cap "
                  f"{ARM[pan['instruct']]['cap']}, {ARM[pan['instruct']]['n_chains']} chains, "
                  f"{CAP_GATE[pan['instruct']]['truncated_rate']:.0%} cut off\n"
                  f"distill   T={ARM[pan['distill']]['temperature']}, cap "
                  f"{ARM[pan['distill']]['cap']}, "
                  f"{CAP_GATE[pan['distill']]['truncated_rate']:.0%} cut off")
        ax.text(0.02, 0.98, ledger, transform=ax.transAxes, va="top", ha="left", fontsize=7,
                color="#555", linespacing=1.5)
    fig.suptitle(f"one trained chain against sixteen bought ones, {DATASET}, n={len(items)}",
                 fontsize=10, x=0.02, ha="left", color="#333")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "figure_distill_dividend.png", dpi=160)
    plt.show()

## 13. The verdict

The gates of section 6, read off the run, and the branch of page copy they select.


In [ ]:
VERDICT = {"per_size": [], "branch": None, "copy": None}
if not HAVE_RUN:
    print("nothing generated: no verdict.")
else:
    for pan in PANELS:
        inst, dist = pan["instruct"], pan["distill"]
        rows_at_n = per_item_rows(inst, ARM[inst]["n_chains"])
        oracle_by_item = {pid: r["any_correct"] for pid, r in rows_at_n.items()}
        vote_by_item = {pid: r["plurality_correct"] for pid, r in rows_at_n.items()}
        d_by_item = POINTS[dist]["per_item"]
        vs_oracle = paired_bootstrap(d_by_item, oracle_by_item)
        vs_vote = paired_bootstrap(d_by_item, vote_by_item)
        row = dict(size_b=pan["size_b"], instruct=inst, distill=dist,
                   distill_pass_at_1=POINTS[dist]["pass_at_1"],
                   instruct_oracle=pan["oracle_at_max"], instruct_vote=pan["vote_at_max"],
                   vs_oracle=vs_oracle, vs_vote=vs_vote,
                   dominates=bool(vs_oracle and vs_oracle["ci"][0] > 0),
                   distill_tokens=POINTS[dist]["mean_tokens_per_chain"],
                   instruct_tokens=pan["tokens_at_max"], cost_ratio=pan["cost_ratio"],
                   oracle_reaches_distill_at_n=pan["oracle_reaches_distill_at_n"])
        VERDICT["per_size"].append(row)

    harness.table(
        "Gate 1, dominance: one distill chain against a perfect judge over the pile",
        ["size", "distill pass@1", "instruct oracle", "difference", "95% interval", "p",
         "dominates"],
        [[size_label(r["size_b"]), f"{r['distill_pass_at_1']:.0%}",
          f"{r['instruct_oracle']:.0%}",
          f"{r['vs_oracle']['delta'] * 100:+.1f} pts",
          f"[{r['vs_oracle']['ci'][0] * 100:+.1f}, {r['vs_oracle']['ci'][1] * 100:+.1f}]",
          f"{r['vs_oracle']['p']:.4f}", "yes" if r["dominates"] else "no"]
         for r in VERDICT["per_size"]],
        note=f"Item-level bootstrap, {BOOTSTRAP_REPS:,} resamples, seeded. Nothing selects "
             f"among the distill's chains; the instruct side is given the best case a judge "
             f"could reach.")

    harness.table(
        "Gate 2, cost: tokens the model wrote per problem",
        ["size", "distill, one chain", "instruct, the whole pile", "ratio",
         "oracle reaches the distill at"],
        [[size_label(r["size_b"]), f"{r['distill_tokens']:,.0f}",
          f"{r['instruct_tokens']:,.0f}",
          f"{r['cost_ratio']:.1f}x" if r["cost_ratio"] else "-",
          f"N={r['oracle_reaches_distill_at_n']}"
          if r["oracle_reaches_distill_at_n"] else "never, on this axis"]
         for r in VERDICT["per_size"]])

    cap_failures = [k for k, v in CAP_GATE.items() if not v["passes"]]
    print(f"\ngate 3, truncation: " +
          ("every arm under the threshold" if not cap_failures
           else f"FAILED on {', '.join(cap_failures)}. Raise the cap and re-run; these arms "
                f"measure the cap."))

    ours = {a["size_b"]: POINTS[a["key"]]["pass_at_1"] for a in DISTILL_ARMS}
    DIRECTION = dict(
        ours={str(k): v for k, v in ours.items()},
        published={str(k): v["math500_pass1"] for k, v in PUBLISHED.items()},
        reproduced=ours[7.0] > ours[1.5])
    print(f"gate 4, direction: DeepSeek publish 1.5B below 7B on both their benchmarks; here "
          f"{ours[1.5]:.0%} and {ours[7.0]:.0%} "
          f"-> {'reproduced' if DIRECTION['reproduced'] else 'REVERSED'}")

    n_dom = sum(r["dominates"] for r in VERDICT["per_size"])
    VERDICT["branch"] = ("dominance" if n_dom == len(VERDICT["per_size"])
                         else "reversal" if n_dom == 0 else "mixed")
    VERDICT["copy"] = COPY[VERDICT["branch"]]
    VERDICT["cap_gate_passes"] = not cap_failures
    VERDICT["direction"] = DIRECTION
    print(f"\nBRANCH: {VERDICT['branch']}\n\n  " + VERDICT["copy"].replace(". ", ".\n  "))
    if not VERDICT["cap_gate_passes"]:
        print("\n!! The cap gate failed, so no branch may ship: an arm that hit its cap on "
              "more\n   than the threshold measured our budget. Raise the cap, re-run, and "
              "read this again.")

## 14. Independent verification

Everything above was computed from objects in memory. This re-reads `traces.jsonl` from disk and
recomputes the headline numbers from the raw generations alone. If the two disagree the figure
is wrong, and the notebook says so rather than shipping it.


In [ ]:
CHECKED = {}
if not (HAVE_RUN and TRACES_PATH.exists()):
    print("no trace file on disk: nothing to verify against.")
else:
    raw = [r for r in read_traces(TRACES_PATH) if r["run_id"] == RUN_ID
           and r["problem_id"] in set(ITEM_IDS)]
    by_arm = {}
    for r in raw:
        by_arm.setdefault(r["arm"], {}).setdefault(r["problem_id"], []).append(r)

    # Grade again from the trace text rather than reading the stored verdict, so a stored
    # `correct` that drifted from the grader shows up here instead of on the page.
    rkey = lambda r: (r["arm"], r["problem_id"], r["sample_index"])
    regraded = {rkey(r): bool(grade_answer(extract(r["trace"]), r["ground_truth"]))
                for r in raw}
    disagree = [r for r in raw if regraded[rkey(r)] != bool(r["correct"])]
    print(f"  regraded {len(raw):,} chains from their text: {len(disagree)} disagree with "
          f"the stored verdict")
    assert not disagree, "the stored correctness does not match a fresh grading"

    for a in ARMS:
        rs_by_item = {pid: sorted(v, key=lambda r: r["sample_index"])[:a["n_chains"]]
                      for pid, v in by_arm.get(a["key"], {}).items()}
        n = len(rs_by_item)
        if a["family"] == "distill":
            acc = mean([mean([regraded[rkey(r)] for r in v]) for v in rs_by_item.values()])
            CHECKED[a["key"]] = dict(pass_at_1=acc, n_items=n)
            got = POINTS[a["key"]]["pass_at_1"]
            print(f"  {a['key']:<14} pass@1 {acc:>6.1%}   (in memory {got:>6.1%})")
            assert abs(acc - got) < 1e-9, f"{a['key']} disagrees with the trace file"
        else:
            cov = mean([any(regraded[rkey(r)] for r in v) for v in rs_by_item.values()])
            n_vote = 0
            for v in rs_by_item.values():
                labs, _ = cluster([vote_key(r["trace"]) for r in v])
                c = Counter([l for l in labs if l is not NO_ANSWER]).most_common()
                top = c[0][1] if c else 0
                winners = [x for x, k in c if k == top]
                n_vote += bool(len(winners) == 1
                               and grade_answer(winners[0], v[0]["ground_truth"]))
            CHECKED[a["key"]] = dict(coverage=cov, vote=n_vote / n, n_items=n)
            top_pt = CURVES[a["key"]][-1]
            print(f"  {a['key']:<14} vote {n_vote / n:>6.1%} coverage {cov:>6.1%}   "
                  f"(in memory {top_pt['vote']:>6.1%} / {top_pt['coverage']:>6.1%})")
            assert abs(cov - top_pt["coverage"]) < 1e-9, f"{a['key']} coverage disagrees"
            assert abs(n_vote / n - top_pt["vote"]) < 1e-9, f"{a['key']} vote disagrees"
    print("\nOK   every headline number matches the raw generations exactly")

## 15. Artifacts

`derived.json` is the contract with the page: the page reads it and nothing else, and no
inference runs at demonstration time. Its `claims` block is read before any number.

The manifest keeps pages 5 to 9's shape, so the `PROVENANCE.md` rebuild needs no special case
for this page. Promote a run by copying `derived.json` to `web/data/page-14-derived.json`.


In [ ]:
manifest = {
    "run_id": RUN_ID,
    "notebook": "page_14_distill_dividend.ipynb",
    "notebook_sha256": harness.file_sha256("page_14_distill_dividend.ipynb"),
    "notebook_git_sha": harness.git_sha(),
    "harness": harness.fingerprint(),
    "page": 14,
    "run_mode": RUN_MODE,
    "replayed": RUN_MODE == "REPLAY",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "models": [dict(MODEL_META.get(a["key"], {"id": a["model"]}),
                    arm=a["key"], family=a["family"], size_b=a["size_b"],
                    role="best-of-N baseline" if a["family"] == "instruct"
                         else "one chain, nothing selected")
               for a in ARMS],
    "decoding": {
        "strategy": "sampled; each family at its own documented operating point",
        "arms": {a["key"]: {"temperature": a["temperature"], "top_p": a["top_p"],
                            "cap": a["cap"], "n_chains": a["n_chains"],
                            "group_size": a["group"], "quant": a["quant"]} for a in ARMS},
        "rationale": {"instruct": "T=0.3 is what page 7's temperature sweep chose for voting "
                                  "on this band",
                      "distill": "T=0.6 is DeepSeek's usage recommendation; they document "
                                 "greedy decoding as producing repetition loops, so a greedy "
                                 "arm would measure the loop"},
        "seed_rule": "sha256(SEED, problem_id, config_id, group_index), one per group",
        "batching": "the chains of one problem are one generate call with "
                    "num_return_sequences",
        "think_prefill": "the distills' prompt ends with <think> where the chat template "
                         "does not already open it; those tokens are ours and are not "
                         "charged as generated",
        "forced_answer_decoding": "greedy at every temperature",
    },
    "forcing": {"stem": FORCE_STEM, "answer_allowance": ANSWER_ALLOWANCE,
                "distill_rule": "a chain cut off inside an open <think> block has the block "
                                "closed before the stem is appended"},
    "items": {"source": "MATH-500 (ch03 copy)", "dataset": DATASET, "n": len(items),
              "item_set_sha256": ITEM_SET_HASH, "pinned_hash": PINNED_ITEM_HASH,
              "seed": SEED, "same_as_pages_6_and_7": ITEM_SET_HASH == PINNED_ITEM_HASH,
              "ids": ITEM_IDS},
    "grading": {"extract": "reasoning_from_scratch.ch03.extract_final_candidate",
                "grade": "reasoning_from_scratch.ch03.grade_answer",
                "answer_scope": "the text after the last </think>, or the whole trace when "
                                "there is none"},
    "vote": {"tally_key": "extract_final_candidate(fallback='number_only') on the answer "
                          "scope",
             "equivalence": "reasoning_from_scratch.ch03.grade_answer, both directions",
             "tie_rule": "a tie returns no answer and grades wrong (ch04 "
                         "self_consistency_vote)",
             "axis_rule": f"a point at N is the mean over {N_RESAMPLES} random N-subsets of "
                          f"the same pile",
             "applies_to": "the instruct arms only; nothing selects among a distill's chains"},
    "estimator": {"distill_pass_at_1": "mean over the arm's chains per problem, then over "
                                       "problems",
                  "interval": "Wilson over problems",
                  "paired_test": f"item-level bootstrap, {BOOTSTRAP_REPS} resamples, seed "
                                 f"{SEED}"},
    "prompt_template": render_prompt("{question}"),
    "cost": {"planned_tokens": PLANNED_TOKENS, "planned_hours": PLANNED_HOURS,
             "distill_tokens_assumed": DISTILL_TOKENS_ASSUMED,
             "priced_from": "page 6's measured spend for the instructs; an assumption for "
                            "the distills, replaced by section 10's measurement"},
    "versions": {"torch": torch.__version__,
                 "transformers": __import__("transformers").__version__},
    "integrity": INTEGRITY,
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"manifest.json  {harness.kb(MANIFEST_PATH):>9.1f} KB")

if not HAVE_RUN:
    print("\nPLAN: no run, so no derived.json. The manifest above records the plan itself.")

In [ ]:
if HAVE_RUN:
    derived = {
        "manifest": manifest,
        # Read before any number on this page.
        "claims": {
            # ---- blocking ----------------------------------------------------------
            # The same hundred problems pages 6 to 9 measured, or nothing here is paired.
            "same_items_as_the_measured_spine": ITEM_SET_HASH == PINNED_ITEM_HASH,
            # No arm was cut off more than the threshold section 6 declared.
            "no_arm_was_censored_by_its_cap": VERDICT["cap_gate_passes"],
            # Both panels have both sides.
            "both_sizes_measured": len(PANELS) == 2 and all(
                p["curve"] and p["point"] for p in PANELS),
            # Nothing selected among the distills' chains, which is the page's whole claim
            # about what it is comparing. Structural: no distill arm has a best-of-N curve,
            # and its number is the mean over its chains rather than a pick from them.
            "nothing_selected_for_the_distill": (
                not any(a["key"] in CURVES for a in DISTILL_ARMS)
                and all(POINTS[a["key"]]["n_chains"]
                        == POINTS[a["key"]]["n_items"] * a["n_chains"]
                        for a in DISTILL_ARMS)),

            # ---- reported, never blocking: these decide what the page SAYS -----------
            # The payoff branch: one trained chain past a perfect judge over sixteen.
            "distill_beats_the_oracle_at_both_sizes": VERDICT["branch"] == "dominance",
            "distill_beats_the_oracle_at_some_size": VERDICT["branch"] in ("dominance",
                                                                          "mixed"),
            # The cost half of the payoff: it is also the cheaper of the two.
            "one_distill_chain_is_cheaper_than_the_pile": all(
                r["distill_tokens"] < r["instruct_tokens"] for r in VERDICT["per_size"]),
            # Whether sixteen chains and a perfect judge ever reach the distill at all.
            "the_oracle_never_reaches_the_distill": all(
                r["oracle_reaches_distill_at_n"] is None for r in VERDICT["per_size"]),
            # DeepSeek's published ordering, on our instrument.
            "published_size_direction_reproduces": VERDICT["direction"]["reproduced"],
            # The vote, which is what a reader can actually run, against the distill.
            "distill_beats_the_vote_at_both_sizes": all(
                r["distill_pass_at_1"] > r["instruct_vote"] for r in VERDICT["per_size"]),
        },
        "branch": VERDICT["branch"],
        "branch_copy": VERDICT["copy"],
        "branch_copy_alternatives": COPY,
        "procedure": (
            f"On the {len(items)} MATH-500 level 4-5 problems pages 6 and 7 drew: sample "
            f"{ARM['instruct-1.5B']['n_chains']} chains from each instruct checkpoint at "
            f"T=0.3 under a {ARM['instruct-1.5B']['cap']}-token cap and read them as a "
            f"majority vote and as an oracle over random N-subsets; sample "
            f"{ARM['distill-1.5B']['n_chains']} chains from each distill at T=0.6 under an "
            f"{ARM['distill-1.5B']['cap']}-token cap and average them, selecting nothing. "
            f"Grade every chain with ch03 on the text after its thinking block. Plot "
            f"accuracy against the tokens each method wrote."),
        "panels": PANELS,
        "gates": GATES,
        "verdict": VERDICT,
        "cap_gate": CAP_GATE,
        "published_prediction": PUBLISHED,
        "priors": {"page_6_greedy": P6_GREEDY, "page_6_mean_tokens": P6_SPEND,
                   "page_7_pile": P7_HANDOFF},
        "verification": CHECKED,
    }

    BLOCKING = ("same_items_as_the_measured_spine", "no_arm_was_censored_by_its_cap",
                "both_sizes_measured", "nothing_selected_for_the_distill")
    harness.table(
        "what this run says the page may claim",
        ["claim", "blocking", "value"],
        [[k, "yes" if k in BLOCKING else "", "true" if v else "FALSE"]
         for k, v in derived["claims"].items()],
        note="the page reads this block; a blocking claim that is false is a build error "
             "there, never a conditional in the component")

    # The gate runs BEFORE anything is written: a refused run leaves the artifacts it could
    # not improve on, rather than replacing them with ones the page refuses to load.
    _failed = [k for k in BLOCKING if not derived["claims"][k]]
    assert not _failed, (
        f"blocking claim(s) false: {_failed}. derived.json was not written, so the last good "
        f"run is still on disk. Fix the run rather than the gate.")

    DERIVED_PATH.write_text(json.dumps(derived, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"\nderived.json   {harness.kb(DERIVED_PATH):>9.1f} KB   <- bundled with the page")
    print(f"traces.jsonl   {harness.kb(TRACES_PATH):>9.1f} KB   <- the download")
    print(f"\npromote with:  cp {DERIVED_PATH} ../web/data/page-14-derived.json")
    print(f"branch shipped: {derived['branch']}")

## Notes for re-derivation

- **The two families are not run at one temperature, and that is a decision rather than an
  oversight.** Each runs where its own documentation puts it. A matched-temperature arm is the
  obvious sensitivity check if the result is ever challenged, and it is a second run, not a
  reinterpretation of this one.
- **Nothing selects among a distill's chains.** Four are generated per problem so that pass@1
  is a mean rather than a coin flip. Voting them, or taking the best, would make the distill
  the thing this page argues it does not need to be.
- **The instruct side is given the best case.** Coverage is what a perfect judge would score,
  which no judge page 8 measured reached. Comparing against the vote alone would flatter the
  distill by the size of that gap.
- **Cost is tokens the model wrote.** The forced answer after a cap is our intervention and is
  counted separately; the `<think>` we prefill is ours and is never charged.
- **A cap is a claim.** The distills run at 8,192 and the instructs at page 6's 2,048, and both
  are gated on truncation rather than assumed to be generous. An arm over the threshold is a
  measurement of our budget, and the notebook refuses to publish it.
- **A group is the unit of reproduction.** One seed per group, recorded on every record, so a
  group regenerates exactly. Different hardware, group size or transformers version will
  produce different chains; the shape should survive, the individual traces will not.
- Changing the model, cap, temperature, chain count or item draw changes `RUN_ID`; records
  under other run ids stay in the file and are never replayed.

## What this run may and may not claim

- **Supported:** the accuracy and token cost of four named checkpoints on the 100 MATH-500
  level 4-5 problems pages 6 and 7 drew, graded by ch03, at the precisions the manifest
  records.
- **Supported:** whether one distill chain clears a perfect judge over sixteen instruct chains
  of the same size, paired at the problem level.
- **Not supported:** anything about DeepSeek's published numbers as numbers. They are on other
  populations; only their direction is compared, and only in gate 4.
- **Not supported:** anything about the closed models on this page's other figure, which are
  cited and not measured.
- **Not supported:** a general claim that distillation beats sampling. One band, one item draw,
  four checkpoints, one seed, and 8-bit at 7B.
